In [ ]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import urljoin
import requests

def init_driver(url, headless=False):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--hide-scrollbars")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.get(url)
    driver.maximize_window()

    # Login if prompted
    try:
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, 'username'))).send_keys("fourbrotherstrading@icloud.com")
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, 'password'))).send_keys("Sultanmirza1501@")
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, './/button[text()="Sign in"]'))).click()
        print("✅ Logged in")
        time.sleep(2)
    except:
        print("🔒 Already logged in / no login popup")

    return driver


def scrape_auction(driver, html_dir="live_html", screenshot_dir="screenshots", pdf_dir="inspection_reports"):
    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(screenshot_dir, exist_ok=True)
    os.makedirs(pdf_dir, exist_ok=True)

    previous_lot = ""

    try:
        while True:
            if not driver.window_handles:
                print("🚪 Browser closed. Exiting.")
                break

            try:
                # Wait for Lot <h1>
                try:
                    lot_elem = WebDriverWait(driver, 4).until(
                        EC.presence_of_element_located((By.XPATH, './/div[contains(@class,"col-lg-9")]/h1'))
                    )
                    lot_text = lot_elem.text.strip()
                    lot_number = lot_text.split(":")[0].replace("Lot", "").strip()
                except TimeoutException:
                    time.sleep(1)
                    continue

                if lot_number != previous_lot:
                    previous_lot = lot_number
                    print(f"\n📌 NEW VEHICLE: Lot {lot_number}")

                    # Extract Registration
                    try:
                        reg_element = WebDriverWait(driver, 3).until(
                            EC.presence_of_element_located((
                                By.XPATH,
                                '//table[contains(@class,"vehicle-table")]//th[text()="Registration"]/following-sibling::td'
                            ))
                        )
                        reg_number = reg_element.text.strip().replace(" ", "_")
                    except:
                        reg_number = "UNKNOWN"

                    # Filenames
                    html_file = os.path.join(html_dir, f"{lot_number}_{reg_number}.html")
                    screenshot_file = os.path.join(screenshot_dir, f"{lot_number}_{reg_number}.png")

                    # Save HTML
                    with open(html_file, "w", encoding="utf-8") as f:
                        f.write(driver.page_source)
                    print(f"💾 HTML SAVED: {html_file}")

                    # Save screenshot
                    driver.save_screenshot(screenshot_file)
                    print(f"📸 SCREENSHOT SAVED: {screenshot_file}")

                    # Download Inspection Report
                    try:
                        report_elem = driver.find_element(By.XPATH, './/a[contains(@class,"btn-primary") and contains(text(),"Download")]')
                        report_url = urljoin(driver.current_url, report_elem.get_attribute("href"))
                        pdf_file = os.path.join(pdf_dir, f"{lot_number}_{reg_number}.pdf")
                        r = requests.get(report_url, stream=True)
                        with open(pdf_file, "wb") as f:
                            for chunk in r.iter_content(1024):
                                f.write(chunk)
                        print(f"📄 INSPECTION REPORT SAVED: {pdf_file}")
                    except:
                        print("⚠️ No inspection report found")

                time.sleep(2)

            except StaleElementReferenceException:
                print("♻️ Stale element — retrying...")
                continue

            except Exception as e:
                print(f"⚠️ Temporary issue: {e}")
                time.sleep(2)
                continue

    except KeyboardInterrupt:
        print("⛔ Stopped by user.")
    finally:
        driver.quit()
        print("🚪 Browser closed.")


if __name__ == "__main__":
    url = "https://www.centralcarauctions.com/portal/auction/buyer/webauction/auction/922"
    driver = init_driver(url, headless=False)
    scrape_auction(driver)


✅ Logged in

📌 NEW VEHICLE: Lot 1
💾 HTML SAVED: live_html\1_NG60EXT.html
📸 SCREENSHOT SAVED: screenshots\1_NG60EXT.png
📄 INSPECTION REPORT SAVED: inspection_reports\1_NG60EXT.pdf

📌 NEW VEHICLE: Lot 2
💾 HTML SAVED: live_html\2_SB59UBR.html
📸 SCREENSHOT SAVED: screenshots\2_SB59UBR.png
📄 INSPECTION REPORT SAVED: inspection_reports\2_SB59UBR.pdf

📌 NEW VEHICLE: Lot 3
💾 HTML SAVED: live_html\3_J14LTJ.html
📸 SCREENSHOT SAVED: screenshots\3_J14LTJ.png
📄 INSPECTION REPORT SAVED: inspection_reports\3_J14LTJ.pdf

📌 NEW VEHICLE: Lot 4
💾 HTML SAVED: live_html\4_CE57WSL.html
📸 SCREENSHOT SAVED: screenshots\4_CE57WSL.png
📄 INSPECTION REPORT SAVED: inspection_reports\4_CE57WSL.pdf

📌 NEW VEHICLE: Lot 5
💾 HTML SAVED: live_html\5_SD07WYU.html
📸 SCREENSHOT SAVED: screenshots\5_SD07WYU.png
📄 INSPECTION REPORT SAVED: inspection_reports\5_SD07WYU.pdf

📌 NEW VEHICLE: Lot 6
💾 HTML SAVED: live_html\6_NJ05SVV.html
📸 SCREENSHOT SAVED: screenshots\6_NJ05SVV.png
📄 INSPECTION REPORT SAVED: inspection_reports\6_N

In [ ]:
import os
import csv
from bs4 import BeautifulSoup

def parse_last_html():
    folder = "live_html"
    files = sorted(os.listdir(folder), key=lambda x: os.path.getmtime(os.path.join(folder, x)))
    last_file = os.path.join(folder, files[-1])
    
    print(f"📌 Reading File: {last_file}")
    
    with open(last_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    bid_list = soup.find("ul", id="biddinghistory")
    if not bid_list:
        print("⚠️ No Bidding History Found!")
        return

    items = bid_list.find_all("li")

    results = []
    current_lot = None
    bids = []
    status = ""

    for li in items:
        text = li.get_text(strip=True).lower()
        raw = li.get_text(strip=True)

        if "lot changed:" in text:
            if current_lot:
                last_bid = bids[0] if bids else ""
                results.append([current_lot, bids, status, last_bid])
            current_lot = raw.replace("Lot changed:", "").strip()
            bids = []
            status = ""

        elif "not sold" in text:
            status = "not_sold"

        elif "provisionally" in text and "sold" in text:
            status = "provisionally"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "sold" in text and "not sold" not in text:
            status = "sold"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "progress" in text:
            status = "in_progress"

        elif "bid:" in text:
            if status in ["sold", "provisionally"]:
                price = raw.split("£")[-1].strip()
                bids.append("£" + price)

    # Last lot
    if current_lot:
        last_bid = bids[0] if bids else ""
        results.append([current_lot, bids, status, last_bid])

    # Merge empty lots into previous
    cleaned_results = []
    for lot, bids_list, st, last_bid in results:
        if bids_list or st in ["sold", "provisionally"]:  # keep if it has bids or sold/provisional
            cleaned_results.append([lot, bids_list, st, last_bid])
        else:
            # merge status if previous exists
            if cleaned_results:
                cleaned_results[-1][2] = st  # update status of previous lot

    # Flatten bids list to string
    final_results = []
    for lot, bids_list, st, last_bid in cleaned_results:
        bids_str = ", ".join(bids_list) if bids_list else ""
        final_results.append([lot, bids_str, st, last_bid])

    # Print output
    print("\n=== BIDDING RESULT ===")
    for row in final_results:
        print(row)

    # Save CSV
    csv_file = "CCA_LIVE_data.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot", "Bids", "Bidding Status", "Last Bid"])
        for row in final_results:
            writer.writerow(row)

    print(f"\n💾 CSV Saved Successfully: {csv_file}")


# RUN
parse_last_html()


📌 Reading File: live_html\108_EA69ZPG.html

=== BIDDING RESULT ===
['108', '£18,150, £18,150, £18,100, £18,050, £18,000, £17,900, £17,800, £17,600, £17,400, £17,200, £17,000', 'provisionally', '£18,150']
['107', '£5,500, £5,500, £5,450, £5,400, £5,350, £5,300, £5,200, £5,100, £5,000', 'provisionally', '£5,500']
['106', '£9,850, £9,850, £9,800, £9,750, £9,700, £9,650, £9,600, £9,550, £9,500, £9,450, £9,400, £9,350, £9,300, £9,250, £9,200, £9,100, £9,000, £8,900, £8,800, £8,700, £8,600, £8,500, £8,400, £8,300, £8,200', 'sold', '£9,850']
['105', '£8,700, £8,700, £8,650, £8,600, £8,550, £8,500, £8,450, £8,400, £8,350, £8,300, £8,250, £8,200, £8,100, £8,000, £7,900, £7,800', 'not_sold', '£8,700']
['103', '£13,750, £13,750, £13,700, £13,600, £13,500, £13,400, £13,200, £13,000', 'provisionally', '£13,750']
['102', '£6,050, £6,050, £6,000, £5,950, £5,900, £5,850, £5,800, £5,750, £5,700, £5,650, £5,600, £5,550, £5,500, £5,450, £5,400, £5,350, £5,300, £5,250, £5,200, £5,100, £5,000, £4,900, £4,8

In [ ]:
import os
import csv

def save_lot_reg_csv():
    folder = "live_html"

    files = []
    for f in os.listdir(folder):
        if "_" not in f:
            continue

        lot = f.split("_")[0]
        if lot.isdigit():
            files.append(f)

    # sort safely
    files = sorted(files, key=lambda x: int(x.split("_")[0]))

    csv_file = "lot_reg_mapping.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot Number", "Reg"])

        for file in files:
            lot_number = file.split("_")[0]
            reg_name = file.split("_", 1)[1].replace(".html", "")
            writer.writerow([lot_number, reg_name])

    print(f"\n💾 CSV Saved Successfully: {csv_file}")

# RUN
save_lot_reg_csv()



💾 CSV Saved Successfully: lot_reg_mapping.csv


In [ ]:
import pandas as pd
import os


live_df = pd.read_csv("CCA_LIVE_data.csv")
reg_df = pd.read_csv("lot_reg_mapping.csv")


reg_df["Lot Number"] = reg_df["Lot Number"].astype(int)
live_df = live_df.reset_index(drop=True)


live_df["Reg"] = reg_df["Reg"].tolist()[:len(live_df)]  


final_csv = "CCA_LIVE_data_final.csv"
live_df.to_csv(final_csv, index=False)

os.remove("CCA_LIVE_data.csv")
os.remove("lot_reg_mapping.csv")

print(f"💾 Merged CSV saved as {final_csv} and original files deleted.")


💾 Merged CSV saved as CCA_LIVE_data_final.csv and original files deleted.


In [ ]:
import pandas as pd

# Load CSVs
live_df = pd.read_csv("CCA_LIVE_data_final.csv")
reg_df = pd.read_csv("cca_data.csv")

# Ensure Reg columns are string type
live_df['Reg'] = live_df['Reg'].astype(str)
reg_df['Reg'] = reg_df['Reg'].astype(str)

# Keep only relevant columns from live_df and rename Bids -> Bidding History
live_df = live_df[["Bids", "Bidding Status", "Last Bid", "Reg"]].rename(columns={"Bids": "Bidding History"})

# Merge on Reg, keep all rows from reg_df
merged_df = pd.merge(reg_df, live_df, on="Reg", how="left")  # merge live data at the end

# Remove duplicates if any
merged_df = merged_df.drop_duplicates(subset="Reg")

# Save final CSV
merged_df.to_csv("final_cca.csv", index=False)

print(f"💾 final_caa.csv created successfully! Total records: {len(merged_df)}")


💾 final_caa.csv created successfully! Total records: 125
